# Build Adjacency Matrices

This notebook constructs distance-based and flow-based adjacency matrices for the 200 selected Citi Bike stations.

## 1. Connect Google Drive

Mount Drive and define paths to the processed files.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Load Processed Data

Load station metadata and the hourly demand matrix created in Notebook 01.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("/content/drive/MyDrive/bike_share_stgcn/data/processed")

STATION_ORDER_FILE = PROCESSED_DIR / "top_200_station_order.csv"
HOURLY_DEMAND_FILE = PROCESSED_DIR / "hourly_pickup_demand_top_200.csv"

station_metadata = pd.read_csv(
    STATION_ORDER_FILE,
    dtype={"station_id": str}
)

hourly_demand = pd.read_csv(
    HOURLY_DEMAND_FILE,
    index_col="timestamp",
    parse_dates=True
)

hourly_demand.columns = hourly_demand.columns.astype(str)

print(f"Stations loaded: {len(station_metadata)}")
print(f"Hourly demand matrix shape: {hourly_demand.shape}")
print(f"Station IDs match matrix columns: {station_metadata['station_id'].tolist() == hourly_demand.columns.tolist()}")

display(station_metadata.head())
display(hourly_demand.iloc[:3, :5])

Stations loaded: 200
Hourly demand matrix shape: (744, 200)
Station IDs match matrix columns: True


,station_id,station_name,latitude,longitude,pickup_count
0,6140.05,W 21 St & 6 Ave,40.741740,-73.994156,16789
1,6233.04,Pier 61 at Chelsea Piers,40.746872,-74.008210,16681
2,5329.03,West St & Chambers St,40.717548,-74.013221,16329
3,5788.13,Lafayette St & E 8 St,40.730207,-73.991026,14563
4,6492.08,9 Ave & W 33 St,40.752568,-73.996765,14468


,6140.05,6233.04,5329.03,5788.13,6492.08
timestamp,,,,,
2025-07-01 00:00:00,1,0,2,3,1
2025-07-01 01:00:00,2,0,2,2,0
2025-07-01 02:00:00,0,1,1,0,0


## 3. Calculate Geographic Distances

Calculate pairwise great-circle distances between the 200 station locations.

In [4]:
from sklearn.metrics.pairwise import haversine_distances

EARTH_RADIUS_KM = 6371.0

coordinates_radians = np.radians(
    station_metadata[["latitude", "longitude"]].to_numpy()
)

distance_matrix_km = haversine_distances(coordinates_radians) * EARTH_RADIUS_KM

np.fill_diagonal(distance_matrix_km, 0)

print(f"Distance matrix shape: {distance_matrix_km.shape}")
print(f"Minimum non-zero distance: {distance_matrix_km[distance_matrix_km > 0].min():.3f} km")
print(f"Maximum distance: {distance_matrix_km.max():.3f} km")
print(f"Distance from station 1 to station 2: {distance_matrix_km[0, 1]:.3f} km")

Distance matrix shape: (200, 200)
Minimum non-zero distance: 0.066 km
Maximum distance: 13.584 km
Distance from station 1 to station 2: 1.314 km


## 4. Build Distance-Based Adjacency

Connect each station to its 10 nearest geographic neighbours using a symmetric Gaussian distance kernel.

In [5]:
N_NEIGHBORS = 10

distance_for_neighbors = distance_matrix_km.copy()
np.fill_diagonal(distance_for_neighbors, np.inf)

nearest_distances = np.sort(distance_for_neighbors, axis=1)[:, :N_NEIGHBORS]
sigma_km = nearest_distances.mean()

distance_adjacency = np.zeros_like(distance_matrix_km, dtype=np.float32)

for i in range(len(station_metadata)):
    nearest_indices = np.argsort(distance_for_neighbors[i])[:N_NEIGHBORS]

    for j in nearest_indices:
        weight = np.exp(
            -(distance_matrix_km[i, j] ** 2) / (sigma_km ** 2)
        )

        distance_adjacency[i, j] = weight
        distance_adjacency[j, i] = max(distance_adjacency[j, i], weight)

np.fill_diagonal(distance_adjacency, 0)

degrees = (distance_adjacency > 0).sum(axis=1)
edge_count = np.count_nonzero(np.triu(distance_adjacency > 0, k=1))

print(f"Number of nearest neighbours (k): {N_NEIGHBORS}")
print(f"Sigma (mean 10-nearest-neighbour distance): {sigma_km:.3f} km")
print(f"Distance adjacency shape: {distance_adjacency.shape}")
print(f"Undirected graph edges: {edge_count:,}")
print(
    f"Node degree — minimum: {degrees.min()}, "
    f"mean: {degrees.mean():.2f}, "
    f"maximum: {degrees.max()}"
)
print(
    f"Non-zero edge weights — minimum: "
    f"{distance_adjacency[distance_adjacency > 0].min():.4f}, "
    f"maximum: {distance_adjacency.max():.4f}"
)
print(f"Adjacency is symmetric: {np.allclose(distance_adjacency, distance_adjacency.T)}")
print(f"Self-loops included: {np.any(np.diag(distance_adjacency) != 0)}")

Number of nearest neighbours (k): 10
Sigma (mean 10-nearest-neighbour distance): 0.602 km
Distance adjacency shape: (200, 200)
Undirected graph edges: 1,209
Node degree — minimum: 10, mean: 12.09, maximum: 18
Non-zero edge weights — minimum: 0.0000, maximum: 0.9879
Adjacency is symmetric: True
Self-loops included: False


### Result

The distance graph contains 200 nodes and 1,209 undirected weighted edges. The graph is symmetric, has no self-loops, and connects each station to at least 10 local neighbours.

## 5. Inspect Distance-Based Neighbours

Inspect the geographic neighbours of the first selected station.

In [6]:
station_index = 0

neighbor_indices = np.where(distance_adjacency[station_index] > 0)[0]

neighbor_table = pd.DataFrame({
    "station_id": station_metadata.iloc[neighbor_indices]["station_id"].values,
    "station_name": station_metadata.iloc[neighbor_indices]["station_name"].values,
    "distance_km": distance_matrix_km[station_index, neighbor_indices],
    "edge_weight": distance_adjacency[station_index, neighbor_indices],
}).sort_values("distance_km")

print(f"Selected station: {station_metadata.iloc[station_index]['station_name']}")
print(f"Number of connected neighbours: {len(neighbor_table)}")

display(neighbor_table.round({
    "distance_km": 3,
    "edge_weight": 6,
}))

Selected station: W 21 St & 6 Ave
Number of connected neighbours: 14


,station_id,station_name,distance_km,edge_weight
5,6064.08,W 18 St & 6 Ave,0.228,0.866575
4,6182.02,W 20 St & 7 Ave,0.271,0.816204
10,6215.04,W 25 St & 6 Ave,0.336,0.733176
9,6098.02,W 20 St & 5 Ave,0.345,0.720717
2,6257.03,W 24 St & 7 Ave,0.362,0.697140
1,6098.10,Broadway & E 21 St,0.437,0.591448
0,6173.08,Broadway & W 25 St,0.437,0.590735
3,5989.02,W 15 St & 6 Ave,0.453,0.567823
8,6215.07,W 27 St & 6 Ave,0.492,0.513405
6,5980.11,Broadway & E 19 St,0.516,0.480211


### Result

W 21 St & 6 Ave is connected to 14 nearby stations in the Chelsea and Flatiron area. Edge weights decrease as geographic distance increases, confirming the expected local structure of the distance graph.

## 6. Save Distance Adjacency

Save the distance-based adjacency matrix for later STGCN training.

In [7]:
DISTANCE_ADJACENCY_FILE = PROCESSED_DIR / "adjacency_distance_knn10.npy"
DISTANCE_MATRIX_FILE = PROCESSED_DIR / "pairwise_distance_km.npy"

np.save(DISTANCE_ADJACENCY_FILE, distance_adjacency)
np.save(DISTANCE_MATRIX_FILE, distance_matrix_km)

loaded_distance_adjacency = np.load(DISTANCE_ADJACENCY_FILE)

print(f"Saved distance adjacency to: {DISTANCE_ADJACENCY_FILE}")
print(f"Saved pairwise distances to: {DISTANCE_MATRIX_FILE}")
print(f"Saved matrix shape: {loaded_distance_adjacency.shape}")
print(f"Saved matrix matches original: {np.array_equal(loaded_distance_adjacency, distance_adjacency)}")

Saved distance adjacency to: /content/drive/MyDrive/bike_share_stgcn/data/processed/adjacency_distance_knn10.npy
Saved pairwise distances to: /content/drive/MyDrive/bike_share_stgcn/data/processed/pairwise_distance_km.npy
Saved matrix shape: (200, 200)
Saved matrix matches original: True


## 7. Count Station-to-Station Flows

Count July 2025 trips between pairs of the 200 selected stations.

In [8]:
CLEAN_PICKUPS_FILE = PROCESSED_DIR / "clean_july_2025_pickups.csv"

station_ids = station_metadata["station_id"].astype(str).tolist()
station_to_index = {station_id: i for i, station_id in enumerate(station_ids)}

flow_counts = np.zeros((len(station_ids), len(station_ids)), dtype=np.int64)

CHUNK_SIZE = 200_000
valid_flow_trips = 0

for chunk in pd.read_csv(
    CLEAN_PICKUPS_FILE,
    usecols=["start_station_id", "end_station_id"],
    chunksize=CHUNK_SIZE,
    dtype={"start_station_id": str, "end_station_id": str}
):
    chunk = chunk.dropna(subset=["start_station_id", "end_station_id"])

    selected_flows = chunk[
        chunk["start_station_id"].isin(station_to_index) &
        chunk["end_station_id"].isin(station_to_index)
    ].copy()

    origin_indices = selected_flows["start_station_id"].map(station_to_index).to_numpy()
    destination_indices = selected_flows["end_station_id"].map(station_to_index).to_numpy()

    np.add.at(flow_counts, (origin_indices, destination_indices), 1)

    valid_flow_trips += len(selected_flows)

np.fill_diagonal(flow_counts, 0)

print(f"Flow count matrix shape: {flow_counts.shape}")
print(f"Trips with both endpoints among selected stations: {valid_flow_trips:,}")
print(f"Non-zero directed station pairs: {(flow_counts > 0).sum():,}")
print(f"Largest directed station-to-station flow: {flow_counts.max():,}")

Flow count matrix shape: (200, 200)
Trips with both endpoints among selected stations: 915,594
Non-zero directed station pairs: 35,905
Largest directed station-to-station flow: 676


### Result

Among the selected stations, 915,594 trips had both origin and destination inside the 200-station network. These trips produced 35,905 observed directed station-to-station flow pairs.

## 8. Build Flow-Based Adjacency

Connect stations using the 10 strongest historical trip-flow relationships.

In [9]:
N_FLOW_NEIGHBORS = 10

undirected_flow_counts = flow_counts + flow_counts.T
np.fill_diagonal(undirected_flow_counts, 0)

flow_for_neighbors = undirected_flow_counts.copy()
np.fill_diagonal(flow_for_neighbors, -1)

flow_adjacency = np.zeros_like(undirected_flow_counts, dtype=np.float32)

for i in range(len(station_metadata)):
    strongest_indices = np.argsort(flow_for_neighbors[i])[-N_FLOW_NEIGHBORS:]

    for j in strongest_indices:
        if undirected_flow_counts[i, j] > 0:
            weight = undirected_flow_counts[i, j] / undirected_flow_counts.max()
            flow_adjacency[i, j] = weight
            flow_adjacency[j, i] = max(flow_adjacency[j, i], weight)

np.fill_diagonal(flow_adjacency, 0)

flow_degrees = (flow_adjacency > 0).sum(axis=1)
flow_edge_count = np.count_nonzero(np.triu(flow_adjacency > 0, k=1))

print(f"Flow adjacency shape: {flow_adjacency.shape}")
print(f"Strongest undirected flow count: {undirected_flow_counts.max():,}")
print(f"Undirected graph edges: {flow_edge_count:,}")
print(
    f"Node degree — minimum: {flow_degrees.min()}, "
    f"mean: {flow_degrees.mean():.2f}, "
    f"maximum: {flow_degrees.max()}"
)
print(
    f"Non-zero edge weights — minimum: "
    f"{flow_adjacency[flow_adjacency > 0].min():.4f}, "
    f"maximum: {flow_adjacency.max():.4f}"
)
print(f"Adjacency is symmetric: {np.allclose(flow_adjacency, flow_adjacency.T)}")
print(f"Self-loops included: {np.any(np.diag(flow_adjacency) != 0)}")

Flow adjacency shape: (200, 200)
Strongest undirected flow count: 1,158
Undirected graph edges: 1,381
Node degree — minimum: 10, mean: 13.81, maximum: 39
Non-zero edge weights — minimum: 0.0069, maximum: 1.0000
Adjacency is symmetric: True
Self-loops included: False


### Result

The flow graph contains 1,381 undirected weighted edges. It is symmetric, has no self-loops, and represents the strongest historical trip relationships among the selected stations.

## 9. Inspect Flow-Based Neighbours

Inspect the strongest trip-flow neighbours of the first selected station.

In [10]:
station_index = 0

flow_neighbor_indices = np.where(flow_adjacency[station_index] > 0)[0]

flow_neighbor_table = pd.DataFrame({
    "station_id": station_metadata.iloc[flow_neighbor_indices]["station_id"].values,
    "station_name": station_metadata.iloc[flow_neighbor_indices]["station_name"].values,
    "distance_km": distance_matrix_km[station_index, flow_neighbor_indices],
    "combined_trip_flow": undirected_flow_counts[station_index, flow_neighbor_indices],
    "edge_weight": flow_adjacency[station_index, flow_neighbor_indices],
}).sort_values("combined_trip_flow", ascending=False)

print(f"Selected station: {station_metadata.iloc[station_index]['station_name']}")
print(f"Number of connected flow neighbours: {len(flow_neighbor_table)}")

display(flow_neighbor_table.round({
    "distance_km": 3,
    "edge_weight": 6,
}))

Selected station: W 21 St & 6 Ave
Number of connected flow neighbours: 38


,station_id,station_name,distance_km,combined_trip_flow,edge_weight
22,6266.06,9 Ave & W 22 St,0.780,712,0.614853
15,6306.06,W 22 St & 10 Ave,1.046,641,0.553541
30,6224.05,W 20 St & 8 Ave,0.531,440,0.379965
0,6233.04,Pier 61 at Chelsea Piers,1.314,397,0.342832
18,6182.02,W 20 St & 7 Ave,0.271,392,0.338515
16,5989.02,W 15 St & 6 Ave,0.453,391,0.337651
20,6459.04,10 Ave & W 28 St,1.182,345,0.297927
8,6072.06,Greenwich Ave & 8 Ave,0.776,343,0.296200
14,6072.11,8 Ave & W 16 St,0.641,337,0.291019
35,6224.03,W 22 St & 8 Ave,0.538,320,0.276339


### Result

W 21 St & 6 Ave has 38 flow-based neighbours. Its strongest connection is 9 Ave & W 22 St with 712 combined trips, showing that historical trip flows capture mobility relationships beyond immediate geographic proximity.

## 10. Save Flow Adjacency

Save the flow-based adjacency matrix for later STGCN training and comparison.

In [11]:
FLOW_ADJACENCY_FILE = PROCESSED_DIR / "adjacency_flow_knn10.npy"
FLOW_COUNTS_FILE = PROCESSED_DIR / "flow_counts_top_200.npy"

np.save(FLOW_ADJACENCY_FILE, flow_adjacency)
np.save(FLOW_COUNTS_FILE, flow_counts)

loaded_flow_adjacency = np.load(FLOW_ADJACENCY_FILE)

print(f"Saved flow adjacency to: {FLOW_ADJACENCY_FILE}")
print(f"Saved directed flow counts to: {FLOW_COUNTS_FILE}")
print(f"Saved matrix shape: {loaded_flow_adjacency.shape}")
print(f"Saved matrix matches original: {np.array_equal(loaded_flow_adjacency, flow_adjacency)}")

Saved flow adjacency to: /content/drive/MyDrive/bike_share_stgcn/data/processed/adjacency_flow_knn10.npy
Saved directed flow counts to: /content/drive/MyDrive/bike_share_stgcn/data/processed/flow_counts_top_200.npy
Saved matrix shape: (200, 200)
Saved matrix matches original: True
